# FMP 최신 발표 분기 기준 YoY / QoQ 스크리너 (패치)

기존 `F.yoy_screen` / `F.qoq_screen` 은 `asof=None` 일 때 **전 종목 공통의 '완전 적재 분기'** 를 찾아 뒤로 후퇴한다.
어닝시즌 중간이거나 회계연도가 다른 기업이 섞여 있으면 최신 분기는 커버리지 100% 가 절대 안 나오므로,
이미 2026Q2 를 발표한 기업까지 2026Q1 vs 2025Q1 로 비교되는 문제가 생긴다.

**이 노트북의 방식**
- 종목별로 **자기 자신의 최신 유효 분기 t** 를 잡아 t-4(YoY) / t-1(QoQ) 매칭
- `max_lag`: 전체 최신 분기 대비 지연 허용 분기 수. 1 이면 상장폐지·피인수로 데이터가 멈춘 종목 제외
- `fx_guard`: 최근 분기 값이 통째로 수배~수십 배 점프한 종목(보고통화 변경 의심: YPF ARS, TKC TRY, SKM KRW 등) 제외

**선행 조건 (중요)**
1. `python US_FMP_FS_1_RUN_UPDATE.py` 로 DB 갱신 (기존 max_date 2026-08-02 에 멈춰 있었음)
2. `fmp_panel_cache.parquet` 삭제 후 패널 재구축
3. 본 노트북은 **단독 실행 가능**: 아래 환경 셀이 panel 을 직접 구축한다.
   같은 커널에 panel 이 이미 있으면 그대로 재사용.


In [1]:
# ---- 환경 + 패널 구축 (단독 실행 지원) ----
# 같은 커널에서 기존 노트북을 먼저 돌려 panel 이 이미 있으면 이 셀은 그대로 통과한다.
import sys
from pathlib import Path
import pandas as pd

if 'panel' not in globals():
    def add_repo_path():
        for parent in [Path.cwd()] + list(Path.cwd().parents):
            if (parent / 'DATA').exists():
                if str(parent) not in sys.path:
                    sys.path.insert(0, str(parent))
                return parent
        raise FileNotFoundError('DATA 폴더를 찾을 수 없습니다')

    PROJECT_ROOT = add_repo_path()
    sys.path.insert(0, str(Path.cwd()))

    from DATA import config
    from US_Market.analysis.FS_Data_Analysis_FMP.old_files import FMP_fs_analyzer_v1 as F

    engine = config.get_engine(config.get_db_info())

    START       = '2019-01-01'
    TICKERS     = None
    NAME_TABLE  = None
    PANEL_CACHE = Path('fmp_panel_cache.parquet')   # 데이터 재적재 후에는 이 파일 삭제!

    if PANEL_CACHE.exists() and TICKERS is None:
        panel = pd.read_parquet(PANEL_CACHE)
        panel['q'] = pd.PeriodIndex(panel['q'], freq='Q')
        print(f'cache load: {PANEL_CACHE} ({len(panel):,} rows)')
    else:
        panel = F.build_panel(engine, tickers=TICKERS, start=START, name_table=NAME_TABLE)
        if TICKERS is None:
            panel.assign(q=panel['q'].astype(str)).to_parquet(PANEL_CACHE, index=False)
            print(f'cache save: {PANEL_CACHE}')

TOP_N = globals().get('TOP_N', 100)
UNIT  = globals().get('UNIT', 1e6)

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print(panel.shape, panel['ticker'].nunique(), 'tickers |', 'max_q:', panel['q'].max())


cache load: fmp_panel_cache.parquet (2,204,476 rows)
(2204476, 8) 1988 tickers | max_q: 2026Q3


## 패치 함수 정의

In [2]:
# ==========================================================
# FMP_fs_analyzer_v1_latest_patch.py
# 기업별 "가장 최근 발표된" 분기 기준 YoY / QoQ 스크리너
#
# 기존 F.yoy_screen / F.qoq_screen 의 문제:
#   - asof=None 이면 전 종목 공통의 '완전 적재 분기'를 찾아 뒤로 후퇴
#     -> 어닝시즌 중간(부분 적재)이면 이미 2026Q2 를 발표한 기업까지
#        2026Q1 vs 2025Q1 로 비교됨
# 이 패치:
#   - 종목별로 자기 자신의 최신 분기 t 를 잡고 t-4(YoY) / t-1(QoQ) 매칭
#   - max_lag 로 오래 전에 멈춘 종목(상장폐지/피인수 등) 제외
#   - fx_guard 로 보고통화 변경(YPF ARS, TKC TRY 등) 의심 종목 필터
#
# 사용법 (노트북에서):
#   import FMP_fs_analyzer_v1_latest_patch as FL
#   FL.asof_coverage(panel)                                  # 분기별 커버리지 진단
#   FL.latest_screen(panel, '매출액',  mode='yoy', n=TOP_N, unit=UNIT, min_base=1e7)
#   FL.latest_screen(panel, '영업이익', mode='qoq', n=TOP_N, unit=UNIT)
# ==========================================================
import pandas as pd


# ---------- 내부: 패널 컬럼 자동 탐지 ----------
def _detect_cols(panel):
    cols = set(panel.columns)
    need = {'ticker', 'q', 'concept'}
    if not need.issubset(cols):
        raise ValueError(f'panel 에 {need - cols} 컬럼이 없습니다. columns={list(panel.columns)}')
    val_col = None
    for c in ('value', 'val', 'amount', 'amt'):
        if c in cols:
            val_col = c
            break
    if val_col is None:
        num = [c for c in panel.columns
               if c not in ('ticker', 'q', 'concept', 'company_name', 'date', 'sj_div', 'item')
               and pd.api.types.is_numeric_dtype(panel[c])]
        if len(num) != 1:
            raise ValueError(f'값 컬럼을 특정할 수 없습니다. 후보={num}')
        val_col = num[0]
    name_col = 'company_name' if 'company_name' in cols else None
    date_col = 'date' if 'date' in cols else None
    return val_col, name_col, date_col


def _wide(panel, concept, val_col):
    sub = panel.loc[panel['concept'] == concept]
    if sub.empty:
        raise ValueError(f"concept '{concept}' 이(가) panel 에 없습니다.")
    w = (sub.pivot_table(index='ticker', columns='q', values=val_col, aggfunc='last')
            .sort_index(axis=1))
    return w


# ---------- 진단: 분기별 적재 커버리지 ----------
def asof_coverage(panel, concept='매출액', tail=8):
    """분기별로 값이 있는 종목 수 / 비율. 자동 asof 가 왜 뒤로 후퇴했는지 확인용."""
    val_col, _, _ = _detect_cols(panel)
    w = _wide(panel, concept, val_col)
    total = w.shape[0]
    cov = w.notna().sum(axis=0).to_frame('n_ticker')
    cov['coverage_%'] = (cov['n_ticker'] / total * 100).round(1)
    return cov.tail(tail)


# ---------- 보고통화 변경 의심 탐지 ----------
def _fx_suspect(row_series, lookback=6, jump=8.0):
    """최근 lookback 분기 내 인접 분기 |값| 비율이 jump 배를 넘으면 통화변경/재표시 의심."""
    s = row_series.dropna().astype(float).abs()
    s = s[s > 0].tail(lookback)
    if len(s) < 2:
        return False
    ratio = (s / s.shift(1)).dropna()
    return bool(((ratio > jump) | (ratio < 1.0 / jump)).any())


# ---------- 메인: 종목별 최신 분기 기준 스크리너 ----------
def latest_screen(panel, concept, mode='yoy', n=100, unit=1e6, min_base=None,
                  max_lag=1, fx_guard=True, fx_jump=8.0, ascending=False):
    """
    종목별 최신 분기 t 기준 YoY(t vs t-4) / QoQ(t vs t-1) 상위 N.

    max_lag  : 전체 최신 분기(global max q) 대비 허용 지연 분기 수.
               1 이면 '전체 최신 분기' 또는 '그 직전 분기'가 최신인 종목만 포함
               (상장폐지·피인수로 데이터가 멈춘 종목 제거).
    min_base : 기준값(분모) 하한, USD 원단위 (1e7 = $10M).
    fx_guard : 최근 분기 값이 통째로 수배~수십 배 점프한 종목(보고통화 변경 의심) 제외.
    """
    assert mode in ('yoy', 'qoq')
    lag = 4 if mode == 'yoy' else 1

    val_col, name_col, date_col = _detect_cols(panel)
    w = _wide(panel, concept, val_col)          # index=ticker, columns=q (오름차순)
    qs = list(w.columns)
    q_pos = {q: i for i, q in enumerate(qs)}
    global_max = qs[-1]

    # 종목별 최신 유효 분기
    notna = w.notna()
    last_q = notna.apply(lambda r: r[r].index[-1] if r.any() else pd.NaT, axis=1)

    rows = []
    for tkr, tq in last_q.items():
        if pd.isna(tq):
            continue
        # 전체 최신 분기 대비 지연 필터 (멈춘 종목 제거)
        if (global_max - tq).n > max_lag:
            continue
        i = q_pos[tq]
        if i - lag < 0:
            continue
        bq = qs[i - lag]
        t_val = w.at[tkr, tq]
        b_val = w.at[tkr, bq]
        if pd.isna(t_val) or pd.isna(b_val):
            continue
        if min_base is not None and abs(b_val) < min_base:
            continue
        if b_val == 0:
            continue
        if fx_guard and _fx_suspect(w.loc[tkr], jump=fx_jump):
            continue
        growth = (t_val - b_val) / abs(b_val) * 100
        rows.append((tkr, bq, tq, b_val / unit, t_val / unit, growth))

    tag = 't-4' if mode == 'yoy' else 't-1'
    out = pd.DataFrame(rows, columns=['ticker', 'base_q', 't_q',
                                      f'{concept}({tag})', f'{concept}(t)', 'growth_%'])

    # 기업명 매핑
    if name_col:
        nm = (panel[['ticker', name_col]].dropna().drop_duplicates('ticker')
              .set_index('ticker')[name_col])
        out.insert(1, 'company_name', out['ticker'].map(nm).fillna(''))

    out = (out.sort_values('growth_%', ascending=ascending)
              .head(n).reset_index(drop=True))
    out.attrs['global_max_q'] = str(global_max)
    return out


def turnaround_latest(panel, mode='yoy', unit=1e6, max_lag=1, fx_guard=True):
    """종목별 최신 분기 기준 영업이익 흑자전환 (base<0 → t>0)."""
    scr = latest_screen(panel, '영업이익', mode=mode, n=10**9, unit=unit,
                        min_base=None, max_lag=max_lag, fx_guard=fx_guard)
    tag = 't-4' if mode == 'yoy' else 't-1'
    b, t = f'영업이익({tag})', '영업이익(t)'
    out = scr[(scr[b] < 0) & (scr[t] > 0)].copy()
    out['swing'] = out[t] - out[b]
    return out.sort_values('swing', ascending=False).reset_index(drop=True)


## 0. 진단: 분기별 적재 커버리지
자동 `asof` 가 왜 뒤로 후퇴했는지 확인. 최신 1~2개 분기의 coverage_% 가 낮으면
공통 분기 방식으로는 그 분기를 못 쓴다는 뜻.

In [3]:
asof_coverage(panel)

,n_ticker,coverage_%
q,,
2024Q4,1840,92.60
2025Q1,1796,90.40
2025Q2,1820,91.60
2025Q3,1770,89.10
2025Q4,1747,87.90
2026Q1,1687,84.90
2026Q2,891,44.80
2026Q3,19,1.00


## 1. YoY 상위 N (종목별 최신 분기 기준)
`min_base` USD: 기준값 하한 (1e7 = $10M). `t_q` 컬럼이 종목마다 다른 것이 정상.

In [11]:
res = latest_screen(panel, '매출액', mode='yoy', n=TOP_N, unit=UNIT, min_base=1e7)
print('global_max_q:', res.attrs['global_max_q'])
res.to_excel(r'C:\Users\82108\OneDrive\INVESTMENT\미국주식\FS_data_analysis\revenue_growth_ranking_aug\rev_growth_rank.xlsx')

global_max_q: 2026Q3


In [12]:
op = latest_screen(panel, '영업이익', mode='yoy', n=TOP_N, unit=UNIT, min_base=5e6)
op.to_excel(r'C:\Users\82108\OneDrive\INVESTMENT\미국주식\FS_data_analysis\revenue_growth_ranking_aug\op_growth_rank.xlsx')

## 2. QoQ 상위 N (종목별 최신 분기 기준)

In [6]:
latest_screen(panel, '매출액', mode='qoq', n=TOP_N, unit=UNIT, min_base=1e7)

,ticker,company_name,base_q,t_q,매출액(t-1),매출액(t),growth_%
0,ARR,,2026Q1,2026Q2,55.96,210.79,276.70
1,IVR,,2026Q1,2026Q2,24.70,64.18,159.84
2,AGIO,,2026Q1,2026Q2,20.75,44.74,115.68
3,MT,,2026Q1,2026Q2,"15,457.00","32,249.68",108.64
4,LYV,,2026Q1,2026Q2,"3,793.03","7,666.86",102.13
...,...,...,...,...,...,...,...
95,VMC,,2026Q1,2026Q2,"1,755.90","2,155.80",22.77
96,LSTR,,2026Q1,2026Q2,"1,171.29","1,432.26",22.28
97,DHI,,2026Q1,2026Q2,"7,558.10","9,227.10",22.08
98,PII,,2026Q1,2026Q2,"1,658.70","2,022.80",21.95


In [7]:
latest_screen(panel, '영업이익', mode='qoq', n=TOP_N, unit=UNIT, min_base=5e6)

,ticker,company_name,base_q,t_q,영업이익(t-1),영업이익(t),growth_%
0,PLAY,,2026Q1,2026Q2,-9.70,54.20,658.76
1,AMC,,2026Q1,2026Q2,-44.60,238.10,633.86
2,BLDR,,2026Q1,2026Q2,17.92,128.51,617.28
3,CVX,,2026Q1,2026Q2,"3,240.00","21,475.00",562.81
4,MEOH,,2026Q1,2026Q2,83.36,517.47,520.78
...,...,...,...,...,...,...,...
95,LOGI,,2026Q1,2026Q2,136.01,258.55,90.09
96,TAL,,2026Q1,2026Q2,72.98,137.20,87.99
97,SAIA,,2026Q1,2026Q2,66.81,125.21,87.43
98,SHEN,,2026Q1,2026Q2,-8.03,-1.02,87.35


## 3. 영업이익 흑자전환 (종목별 최신 분기 기준)

In [8]:
turnaround_latest(panel, mode='yoy', unit=UNIT)

,ticker,company_name,base_q,t_q,영업이익(t-4),영업이익(t),growth_%,swing
0,STM,,2025Q2,2026Q2,-25.33,220.00,968.58,245.33
1,JACK,,2025Q2,2026Q2,-157.08,38.60,124.57,195.67
2,CVI,,2025Q2,2026Q2,-100.00,78.00,178.00,178.00
3,GME,,2025Q2,2026Q2,-10.80,143.30,"1,426.85",154.10
4,MAN,,2025Q2,2026Q2,-25.30,112.00,542.69,137.30
5,IVR,,2025Q2,2026Q2,-23.33,90.31,487.16,113.64
6,AAP,,2025Q2,2026Q2,-13.00,99.00,861.54,112.00
7,KWR,,2025Q2,2026Q2,-52.51,48.72,192.78,101.23
8,AEO,,2025Q2,2026Q2,-68.06,28.23,141.47,96.29
9,SHOO,,2025Q2,2026Q2,-40.26,39.32,197.67,79.58


In [9]:
turnaround_latest(panel, mode='qoq', unit=UNIT)

,ticker,company_name,base_q,t_q,영업이익(t-1),영업이익(t),growth_%,swing
0,OVV,,2026Q1,2026Q2,-754.00,994.00,231.83,"1,748.00"
1,CVI,,2026Q1,2026Q2,-326.00,78.00,123.93,404.00
2,DXC,,2026Q1,2026Q2,-89.00,241.00,370.79,330.00
3,AMC,,2026Q1,2026Q2,-44.60,238.10,633.86,282.70
4,TEX,,2026Q1,2026Q2,-82.00,187.00,328.05,269.00
5,PII,,2026Q1,2026Q2,-55.20,132.00,339.13,187.20
6,GVA,,2026Q1,2026Q2,-34.08,125.57,468.44,159.65
7,ENPH,,2026Q1,2026Q2,-29.64,51.52,273.80,81.16
8,SCHL,,2026Q1,2026Q2,-26.90,52.90,296.65,79.80
9,ALKS,,2026Q1,2026Q2,-48.28,18.38,138.07,66.66


## 4. 통화변경 의심 종목 확인 (선택)
`fx_guard=False` 로 돌린 결과와 비교하면 어떤 종목이 걸러졌는지 볼 수 있다.

In [10]:
with_fx  = latest_screen(panel, '매출액', mode='yoy', n=10**9, unit=UNIT, min_base=1e7)
without  = latest_screen(panel, '매출액', mode='yoy', n=10**9, unit=UNIT, min_base=1e7, fx_guard=False)
excluded = sorted(set(without['ticker']) - set(with_fx['ticker']))
print(f'fx_guard 로 제외된 종목 {len(excluded)}개')
without[without['ticker'].isin(excluded)].head(30)

fx_guard 로 제외된 종목 4개


,ticker,company_name,base_q,t_q,매출액(t-4),매출액(t),growth_%
549,KOF,,2025Q2,2026Q2,"72,917.00","76,318.00",4.66
567,PAC,,2025Q2,2026Q2,"10,882.00","11,333.04",4.14
862,NRZ,,2025Q2,2026Q2,419.81,221.76,-47.18
866,ABR,,2025Q2,2026Q2,301.77,23.88,-92.09
